<div style="box-sizing: border-box; width: 100%; background: linear-gradient(135deg, #0f172a 0%, #1e293b 60%, #a16207 100%); border-radius: 14px; padding: 28px 36px; color: #f8fafc; font-family: 'Segoe UI', Helvetica, Arial, sans-serif;">
<div style="font-size: 12px; letter-spacing: 3px; text-transform: uppercase; color: #fb923c; margin-bottom: 8px;">El ruido de la Tierra · Parte 7</div>
<div style="font-size: 32px; font-weight: 800; line-height: 1.1; margin-bottom: 6px;">España</div>
<div style="font-size: 17px; font-weight: 300; color: #cbd5e1;">Terremotos en la península, las islas y su entorno</div>
</div>

Este notebook continúa el análisis de `ejercicio_1_carga.ipynb` y trabaja sobre la colección `terremotos` ya cargada y limpiada en él. Si la base de datos no existe, hay que ejecutar antes ese notebook.

A diferencia de las secciones anteriores, aquí usamos **consultas de filtrado con `find()`** (sin `aggregate`): filtros por campo, rangos de fechas, expresiones regulares, campos de subdocumentos, proyecciones, ordenación y operadores geoespaciales (`$near`, `$geoWithin`) apoyados en el índice `2dsphere`.

In [ ]:
from pymongo import MongoClient
from datetime import datetime
import numpy as np
import pandas as pd
import plotly.express as px

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)
col = client["terremotos"]["terremotos"]
print("Documentos en la colección:", col.count_documents({}))

# **7. España**

España tiene una sismicidad **moderada**. Está en el borde entre la placa **euroasiática** y la **africana**, que se acercan unos 4–5 mm al año, mucho más despacio que las placas del Pacífico. La tensión se reparte entre muchas fallas pequeñas en lugar de concentrarse en una gran zona de subducción. Por eso los terremotos frecuentes son de magnitud 3–4 y los de magnitud 5 o más son raros. Las zonas con más actividad son:

- **Sur y sureste peninsular** (Granada, Almería, Murcia, Alicante): las cordilleras Béticas, levantadas por el empuje de África.
- **Mar de Alborán y norte de Marruecos**: el tramo más activo del límite entre placas cerca de España.
- **Golfo de Cádiz y Cabo de San Vicente**: donde se originó el **terremoto de Lisboa de 1755**, cuyo tsunami golpeó Cádiz y Huelva.
- **Pirineos** y **Canarias**, donde la sismicidad está ligada al vulcanismo.

Los terremotos más dañinos de la historia reciente de España no fueron grandes, sino superficiales y cercanos a zonas habitadas: el de **Arenas del Rey (Granada) de 1884** causó más de 800 víctimas, y el de **Lorca (Murcia) de 2011**, de solo magnitud 5,1, causó 9.

### **7.1 Terremotos en territorio español**
Empezamos con el filtro más sencillo: igualdad sobre el campo `region`. La proyección muestra solo los campos que interesan y `sort` ordena por fecha. `impacto.intensidad_instrumental` usa la notación de punto para leer un campo del subdocumento.

In [ ]:
filtro_espana = {"region": "Spain"}
print("Terremotos M ≥ 5 en España:", col.count_documents(filtro_espana))

espana = col.find(
    filtro_espana,
    {"_id": 0, "fecha": 1, "lugar": 1, "magnitud": 1, "profundidad_km": 1, "alerta": 1,
     "impacto.personas_lo_sintieron": 1, "impacto.intensidad_instrumental": 1, "url": 1},
).sort("fecha", 1)
pd.json_normalize(list(espana))

**Resultado:** En casi once años solo hay **dos terremotos de magnitud 5 o más con epicentro en España**, los dos en Andalucía oriental y a 10 km de profundidad:
- **14/07/2025, M5,2**, en el mar a 23 km al este de San José (Cabo de Gata, Almería). Al estar en el mar, nadie estuvo cerca del epicentro y no tiene alerta.
- **14/08/2026, M5,0**, a 1 km de **Alhendín**, en el área metropolitana de **Granada**. Es más pequeño, pero ocurrió bajo una zona poblada: alcanzó una intensidad instrumental cercana a VII (daños leves en edificios) y el USGS le asignó **alerta amarilla**.

Es el mismo patrón que vimos en la sección 6 a menor escala: con una magnitud parecida, el terremoto bajo tierra firme y habitada es mucho más peligroso que el del mar.

### **7.2 Los terremotos más cercanos a Madrid**
Dos terremotos son pocos, así que ampliamos la búsqueda. `$near` usa el índice `2dsphere` para devolver los terremotos a menos de una distancia de un punto, **ya ordenados del más cercano al más lejano**. Tomamos el centro de la península (Madrid) y un radio de 800 km, que cubre toda la península, Baleares y el norte de Marruecos y Argelia. `find()` no devuelve la distancia, así que la calculamos en pandas con la fórmula del semiverseno (haversine).

In [ ]:
MADRID = [-3.7038, 40.4168]   # [longitud, latitud]

cercanos = col.find(
    {"ubicacion": {"$near": {
        "$geometry": {"type": "Point", "coordinates": MADRID},
        "$maxDistance": 800_000,   # en metros
    }}},
    {"_id": 0, "fecha": 1, "lugar": 1, "region": 1, "magnitud": 1, "profundidad_km": 1, "ubicacion": 1},
)
df_cercanos = pd.DataFrame(list(cercanos))

lon = np.radians(df_cercanos["ubicacion"].str["coordinates"].str[0])
lat = np.radians(df_cercanos["ubicacion"].str["coordinates"].str[1])
lon0, lat0 = np.radians(MADRID)
a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat0) * np.cos(lat) * np.sin((lon - lon0) / 2) ** 2
df_cercanos["distancia_km"] = (2 * 6371 * np.arcsin(np.sqrt(a))).round(0)

print("Terremotos a menos de 800 km de Madrid:", len(df_cercanos))
df_cercanos.drop(columns="ubicacion")

**Resultado:** En un radio de 800 km hay **18 terremotos**. Tras los dos españoles (Granada a 370 km y Cabo de Gata a 440 km), **12 se concentran en el mar de Alborán**, entre la costa de Granada y Al Hoceïma (Marruecos), a 520–550 km de Madrid. El resto son casos aislados en Portugal (frente a Sines), el norte de Marruecos y Argelia. El mar de Alborán es, con diferencia, la zona sísmica más activa cerca de España. El evento etiquetado como `Strait of Gibraltar` también está en realidad en Alborán, a unos 50 km de Al Hoceïma: el nombre de la región del USGS es aproximado.

### **7.3 La crisis sísmica de Alborán (2016)**
Combinamos tres condiciones en el mismo filtro: igualdad sobre `region`, una expresión regular sobre `lugar` y un rango de fechas con `$gte` y `$lt`.

In [ ]:
filtro_alboran = {
    "region": "Morocco",
    "lugar": {"$regex": "Al Hoce"},
    "fecha": {"$gte": datetime(2016, 1, 1), "$lt": datetime(2016, 4, 1)},
}
print("Terremotos M ≥ 5 en Alborán entre enero y marzo de 2016:", col.count_documents(filtro_alboran))

alboran = col.find(
    filtro_alboran,
    {"_id": 0, "fecha": 1, "lugar": 1, "magnitud": 1, "profundidad_km": 1, "alerta": 1,
     "impacto.personas_lo_sintieron": 1},
).sort("fecha", 1)
pd.json_normalize(list(alboran))

**Resultado:** Entre el 21 de enero y el 15 de marzo de 2016 se registraron **10 terremotos de magnitud 5 o más** en el mismo punto del mar de Alborán. El principal, de **magnitud 6,3** (25/01/2016 a las 04:22 UTC), es el mayor terremoto del periodo cerca de España. Le precedió un sismo de 5,1 cuatro días antes, y en las dos horas siguientes hubo cuatro réplicas de magnitud 5. El terremoto causó daños en **Melilla**, a unos 80 km del epicentro, y se sintió en toda la costa de Andalucía. Aun así quedó en alerta verde, porque el epicentro estaba en el mar.

### **7.4 ¿Cuáles sintió más gente?**
Ahora filtramos por un campo del subdocumento `impacto`: los terremotos a menos de 1.000 km de Madrid que **más de 100 personas** notificaron haber sentido al USGS. `$geoWithin` con `$centerSphere` busca dentro de un círculo cuyo radio se expresa en radianes (distancia / radio de la Tierra). A diferencia de `$near`, no ordena, así que ordenamos nosotros con `sort`.

In [ ]:
RADIO_TIERRA_KM = 6378.1

sentidos = col.find(
    {
        "ubicacion": {"$geoWithin": {"$centerSphere": [MADRID, 1000 / RADIO_TIERRA_KM]}},
        "impacto.personas_lo_sintieron": {"$gt": 100},
    },
    {"_id": 0, "fecha": 1, "lugar": 1, "magnitud": 1, "alerta": 1,
     "impacto.personas_lo_sintieron": 1, "impacto.intensidad_reportada": 1},
).sort("impacto.personas_lo_sintieron", -1)
pd.json_normalize(list(sentidos))

**Resultado:** El terremoto que más gente notó **no es el más fuerte, sino el que estaba más cerca de una gran ciudad**: el de magnitud 5,0 frente a **Sines (Portugal)** del 26/08/2024, con casi 2.900 avisos, porque se sintió en Lisboa y en gran parte de Portugal. El de Alborán de 2016 (M6,3) tuvo solo 122 avisos, y el de Granada de 2026 (M5,0), 126. Hay que tener en cuenta que este campo cuenta los formularios enviados a la web del USGS (*Did You Feel It?*), muy usada en EE. UU. y poco en España. Sirve para comparar terremotos entre sí, no como número real de personas.

### **7.5 Canarias**
Las islas Canarias tienen origen volcánico y sufrieron la erupción de La Palma en 2021. Buscamos terremotos dentro de un polígono que rodea el archipiélago con `$geoWithin` y `$geometry`.

In [ ]:
CANARIAS = {"type": "Polygon", "coordinates": [[
    [-18.5, 27.4], [-13.2, 27.4], [-13.2, 29.6], [-18.5, 29.6], [-18.5, 27.4],
]]}

filtro_canarias = {"ubicacion": {"$geoWithin": {"$geometry": CANARIAS}}}
print("Terremotos M ≥ 5 en Canarias:", col.count_documents(filtro_canarias))
print("Ejemplo:", col.find_one(filtro_canarias))

**Resultado:** **Ninguno.** Durante la erupción de La Palma (septiembre–diciembre de 2021) hubo miles de terremotos, pero ninguno llegó a magnitud 5 en el catálogo del USGS. Los terremotos volcánicos se producen al abrirse paso el magma y liberan mucha menos energía que los tectónicos. Una consulta vacía también es un resultado: `find_one` devuelve `None`.

### **7.6 Mapa de la sismicidad alrededor de España**
Reutilizamos el filtro `$centerSphere` de 1.000 km y pintamos el resultado. En la proyección pedimos las coordenadas con `ubicacion` y las separamos en pandas.

In [ ]:
df_peninsula = pd.DataFrame(list(col.find(
    {"ubicacion": {"$geoWithin": {"$centerSphere": [MADRID, 1000 / RADIO_TIERRA_KM]}}},
    {"_id": 0, "fecha": 1, "lugar": 1, "magnitud": 1, "profundidad_km": 1, "ubicacion": 1},
).sort("magnitud", 1)))
df_peninsula["longitud"] = df_peninsula["ubicacion"].str["coordinates"].str[0]
df_peninsula["latitud"] = df_peninsula["ubicacion"].str["coordinates"].str[1]
df_peninsula["es_espana"] = df_peninsula["lugar"].str.endswith("Spain")

fig = px.scatter_geo(
    df_peninsula, lat="latitud", lon="longitud",
    color="magnitud", size=2 ** (df_peninsula["magnitud"] - 5), size_max=20,
    symbol="es_espana", symbol_map={True: "star", False: "circle"},
    color_continuous_scale="YlOrRd", range_color=(5, 6.5),
    hover_name="lugar", hover_data={"fecha": "|%d/%m/%Y", "magnitud": True, "profundidad_km": True,
                                     "latitud": False, "longitud": False, "es_espana": False},
    title=f"{len(df_peninsula)} terremotos de magnitud ≥ 5 a menos de 1.000 km de Madrid (2016–2026)",
)
fig.update_traces(marker=dict(opacity=0.85, line=dict(width=1, color="#0f172a")))
fig.update_geos(projection_type="mercator", lonaxis_range=[-14, 8], lataxis_range=[33, 45],
                showland=True, landcolor="#1e293b", showocean=True, oceancolor="#0f172a",
                showcountries=True, countrycolor="#475569", coastlinecolor="#64748b",
                showframe=False, bgcolor="#0f172a", resolution=50)
fig.update_layout(height=560, margin=dict(l=0, r=0, t=50, b=0), showlegend=False,
                  paper_bgcolor="#0f172a", font_color="#e2e8f0",
                  coloraxis_colorbar=dict(title="Magnitud"))
fig.show()

**Resultado:** El mapa resume la sección. Los terremotos de magnitud 5 o más rodean la península por el sur siguiendo el **límite entre África y Eurasia**: desde el Golfo de Cádiz y la costa portuguesa, pasando por el mar de Alborán (donde se agrupan casi todos), hasta el norte de Argelia. Las dos estrellas son los únicos epicentros en España, y están justo en el borde de esa franja, en las Béticas. El interior de la península, los Pirineos, Baleares y Canarias no tienen ningún terremoto de magnitud 5 en el periodo.

**Conclusión de la sección:** España tiene pocos terremotos fuertes comparada con las zonas de la sección 4, pero no está a salvo. El riesgo se concentra en el sureste y en el mar de Alborán, y como vimos en la sección 6, lo que convierte un terremoto moderado en una catástrofe es que sea superficial y ocurra bajo una zona poblada, como pasó en Lorca en 2011 o, a menor escala, en Granada en 2026.